# Load Dataset from HF

This will gather the dataset created using the dataset and load it into memory.


In [ ]:
# lib import
import os
from datasets import load_dataset, get_dataset_config_names

# setup
data = {}
source_ds_cache_dir = os.path.join(os.getcwd(), "data", "wikipos")
configs = get_dataset_config_names("whatphiliptrains/wikipos")

for config in configs:
    data[config] = load_dataset(
        "whatphiliptrains/wikipos", config, cache_dir=source_ds_cache_dir
    )

test = data[configs[-1]]

# verification (optional)
print(f"Dataset length: {len(test['train'])}")
print(test["train"][0])
print(test["train"][-1])

# Trustworthiness and Continuity

Using `scikit-learn` the trustworthiness (and continuity) between the dense embedding and the reduced position vector is calculated


In [ ]:
import csv
import faiss
import numpy as np
# from sklearn.manifold import trustworthiness
# from sklearn.neighbors import NearestNeighbors

n = 15
results = {}


# def calculate_continuity(X, X_embedded, n_neighbors=5):
#     """
#     Calculate continuity metric for dimensionality reduction.

#     Continuity measures whether points that are close in the low-dimensional
#     embedding are also close in the high-dimensional space.
#     """
#     # Ensure X is a numpy array
#     if not isinstance(X, np.ndarray):
#         X = np.array(X)
#     if not isinstance(X_embedded, np.ndarray):
#         X_embedded = np.array(X_embedded)

#     # Find nearest neighbors in low-dimensional space
#     nbrs_embedded = NearestNeighbors(n_neighbors=n_neighbors + 1).fit(X_embedded)
#     _, indices_embedded = nbrs_embedded.kneighbors(X_embedded)

#     # Find nearest neighbors in high-dimensional space
#     nbrs_original = NearestNeighbors(n_neighbors=n_neighbors + 1).fit(X)
#     _, indices_original = nbrs_original.kneighbors(X)

#     continuity_sum = 0
#     n_samples = X.shape[0]

#     for i in range(n_samples):
#         # Get k nearest neighbors in embedded space (excluding self)
#         embedded_neighbors = set(indices_embedded[i][1 : n_neighbors + 1])

#         # Get k nearest neighbors in original space (excluding self)
#         original_neighbors = set(indices_original[i][1 : n_neighbors + 1])

#         # Count how many embedded neighbors are also original neighbors
#         intersection = len(embedded_neighbors.intersection(original_neighbors))
#         continuity_sum += intersection / n_neighbors

#     return continuity_sum / n_samples


def large_scale_trust_and_cont(X, X_embedded, n_neighbors=15):
    """
    Calculates both trustworthiness and continuity for large datasets using Faiss.

    This function computes a common, practical approximation of the metrics,
    often referred to as k-NN Purity (Trustworthiness) and k-NN Preservation (Continuity).
    """
    n_samples = X.shape[0]

    X = np.ascontiguousarray(X, dtype=np.float32)
    X_embedded = np.ascontiguousarray(X_embedded, dtype=np.float32)

    # Build Faiss indexes for both datasets
    print("Building Faiss indexes...")
    index_X = faiss.IndexFlatL2(X.shape[1])
    index_X.add(X)
    index_embedded = faiss.IndexFlatL2(X_embedded.shape[1])
    index_embedded.add(X_embedded)

    # Find k+1 nearest neighbors (the first is the point itself)
    print("Searching for nearest neighbors...")
    _, high_dim_neighbors = index_X.search(X, n_neighbors + 1)
    _, low_dim_neighbors = index_embedded.search(X_embedded, n_neighbors + 1)

    # Exclude the point itself from its list of neighbors
    high_dim_neighbors = high_dim_neighbors[:, 1:]
    low_dim_neighbors = low_dim_neighbors[:, 1:]

    # Calculate scores
    print("Calculating scores...")
    trustworthiness_sum = 0.0
    continuity_sum = 0.0

    for i in range(n_samples):
        # Get the neighbor sets for the current point
        original_neighbors = set(high_dim_neighbors[i])
        embedded_neighbors = set(low_dim_neighbors[i])

        # Find the size of the intersection
        intersection_size = len(original_neighbors.intersection(embedded_neighbors))

        # Trustworthiness: How many of the new neighbors are "true"?
        trustworthiness_sum += intersection_size / n_neighbors

        # Continuity: How many of the original neighbors were "preserved"?
        continuity_sum += intersection_size / n_neighbors

    # Average the scores over all samples
    trust = trustworthiness_sum / n_samples
    cont = continuity_sum / n_samples

    return trust, cont


for config in configs:
    current = data[config]["train"]
    pos = np.column_stack([current["x"], current["y"]])
    embeddings = np.array(current["embeddings"])

    # trust = trustworthiness(X=embeddings, X_embedded=pos, n_neighbors=n)
    # continuity = calculate_continuity(X=embeddings, X_embedded=pos, n_neighbors=n)

    trust, cont = large_scale_trust_and_cont(
        X=embeddings, X_embedded=pos, n_neighbors=n
    )

    results[config] = {"trustworthiness": trust, "continuity": cont}
    print(f"[{config}] : trust: {trust:.4f}, continuity: {cont:.4f}")

with open("results.csv", "w", newline="") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(["config", "trustworthiness", "continuity"])
    for config, metrics in results.items():
        writer.writerow([config, metrics["trustworthiness"], metrics["continuity"]])

print("Results written to 'results.csv'")